# Cleaning Application Data

In [133]:
import numpy as np
import pandas as pd
import re

# Load Data

In [134]:
file_path = "/Users/kaylamullen/Desktop/PITNE/Data/Working_2021-2023_App_Data.csv"
data = pd.read_csv(file_path)

# inspect basic structure
print(data.shape)
print(data.columns)
data.head()

# make backup copy
data_raw = data.copy()

(380, 17)
Index(['ID Number', 'Submission Date', 'Application Property Street Address',
       'Application Property Town', 'Source', 'Source 2', 'Age',
       'Race/Ethnicity', 'Disability', 'Current Residence Town',
       'Current Residence State', 'HH Size', 'Dependents', 'HH Type',
       'HH Income', 'HH Assets', 'FTHB Class?'],
      dtype='object')


# Data Cleaning

## Basic Standardization

In [135]:
# Standardize column names (remove whitespace, lowercase)
data.columns = data.columns.str.strip().str.lower().str.replace(" ", "_")

# inspect basic structure
print(data.shape)
print(data.columns)
data.head()

(380, 17)
Index(['id_number', 'submission_date', 'application_property_street_address',
       'application_property_town', 'source', 'source_2', 'age',
       'race/ethnicity', 'disability', 'current_residence_town',
       'current_residence_state', 'hh_size', 'dependents', 'hh_type',
       'hh_income', 'hh_assets', 'fthb_class?'],
      dtype='object')


,id_number,submission_date,application_property_street_address,application_property_town,source,source_2,age,race/ethnicity,disability,current_residence_town,current_residence_state,hh_size,dependents,hh_type,hh_income,hh_assets,fthb_class?
0,150733,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,41,white,NaN,Somerville,MA,1,NaN,NaN,"37,934.00",NaN,NaN
1,439181,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,32,hispanic,NaN,Westford,MA,2,NaN,NaN,"61,000.00",NaN,NaN
2,697399,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,79,black/afr am,NaN,Malden,MA,2,NaN,NaN,"31,275.00","346,000.00",NaN
3,191565,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,NaN,white,NaN,Haverhill,MA,1,NaN,NaN,"20,299.00",NaN,NaN
4,226436,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,26,white,NaN,Nashua,NH,1,NaN,NaN,"41,000.00",NaN,NaN


In [136]:
duplicate_rows = data[data.duplicated(keep=False)]
duplicate_rows

,id_number,submission_date,application_property_street_address,application_property_town,source,source_2,age,race/ethnicity,disability,current_residence_town,current_residence_state,hh_size,dependents,hh_type,hh_income,hh_assets,fthb_class?


In [137]:
# Remove Duplicates
data = data.drop_duplicates()

In [138]:
data.isnull().sum() # sum of how many rows are null within each column

id_number                                0
submission_date                          0
application_property_street_address      0
application_property_town              367
source                                 103
source_2                               374
age                                     93
race/ethnicity                          23
disability                             361
current_residence_town                   4
current_residence_state                  5
hh_size                                  0
dependents                             118
hh_type                                380
hh_income                                2
hh_assets                               22
fthb_class?                            380
dtype: int64

## ID Number

In [139]:
# how many missing id numbers (including null and white space)
data['id_number'].apply(lambda x: pd.isna(x) or str(x).strip() == '').sum()

np.int64(0)

In [140]:
# create a dataframe containing only the applicants who applied multiple times
duplicate_ids = data[data.duplicated(subset=['id_number'], keep=False)]
duplicate_ids.sort_values(by=['id_number', 'submission_date'])

,id_number,submission_date,application_property_street_address,application_property_town,source,source_2,age,race/ethnicity,disability,current_residence_town,current_residence_state,hh_size,dependents,hh_type,hh_income,hh_assets,fthb_class?
73,45679,4/12/2022,"2 Harvest Drive, Unit 104",NaN,MassAccess,NaN,41,White,NaN,Woburn,MA,1,NaN,NaN,"44,000.00","29,000.00",NaN
96,45679,5/24/2022,"4 Harvest Drive, Unit 212",NaN,CHAPA email,NaN,60,White,NaN,Woburn,MA,1,NaN,NaN,"42,000.00","30,000.00",NaN
192,76111,1/31/2023,28 Groveland Commons Way,NaN,boston.gov,NaN,36,hispanic/latino,NaN,Salem,MA,2,1.0,NaN,"53,000.00","3,900.00",NaN
158,76111,12/6/2022,"180 Chickering Road, Unit 307C",NaN,MyMassHome,NaN,53,Hispanic,NaN,Salem,MA,2,1.0,NaN,"53,000.00","7,500.00",NaN
171,159142,12/20/2022,1 Rocky Point,Carlisle,CHAPA,NaN,NaN,White,NaN,Quincy,MA,1,0.0,NaN,"76,000.00","14,000.00",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,969361,4/12/2022,"2 Harvest Drive, Unit 104",NaN,ready buyer list,NaN,NaN,Asian,NaN,Derry,NH,1,NaN,NaN,"55,000.00","71,000.00",NaN
86,975195,5/10/2022,"3 Harvest Drive, Unit 105",NaN,zillow,NaN,NaN,White,NaN,Woburn,MA,1,NaN,NaN,"62,000.00","28,000.00",NaN
95,975195,5/24/2022,"4 Harvest Drive, Unit 212",NaN,zillow,NaN,60,White,NaN,Woburn,MA,1,NaN,NaN,"62,000.00","28,000.00",NaN
122,983151,11/1/2022,3 Dickens Lane,NaN,NaN,NaN,43,White,NaN,Stow,MA,1,0.0,NaN,"19,044.00","250,000.00",NaN


In [141]:
# how many applicants applied multiple times
duplicate_ids['id_number'].nunique()

35

## Submission Date

In [142]:
# how many missing submission dates (including null and white space)
data['submission_date'].apply(lambda x: pd.isna(x) or str(x).strip() == '').sum()

np.int64(0)

## Application Property Street Address

In [143]:
# how many missing app property street addresses (including null and white space)
data['application_property_street_address'].apply(lambda x: pd.isna(x) or str(x).strip() == '').sum()

np.int64(0)

In [144]:
# split street address and unit
unit_pattern = r'(?:,\s*)?\b(?:apt\.?|unit|#)\s*\S+$'

# extract unit number
data['unit_number'] = data['application_property_street_address'].str.extract(f'({unit_pattern})', flags=re.IGNORECASE)

In [145]:
# clean street address column by removing extracted data
data['street_address'] = data['application_property_street_address'].str.replace(f'{unit_pattern}', '', flags=re.IGNORECASE, regex=True).str.strip(", ").str.strip()

In [146]:
# Standardize unit number formatting
data['unit_number'] = data['unit_number'].str.title()

# Remove ", " from unit number column
data['unit_number'] = data['unit_number'].str.lstrip(", ").str.strip()
data[['street_address', 'unit_number']].head(30)  # Show first 20 rows

,street_address,unit_number
0,8 Ciderpress Way,NaN
1,8 Ciderpress Way,NaN
2,8 Ciderpress Way,NaN
3,8 Ciderpress Way,NaN
4,8 Ciderpress Way,NaN
5,8 Ciderpress Way,NaN
6,1301 Albion Road,NaN
7,1301 Albion Road,NaN
8,1301 Albion Road,NaN
9,1301 Albion Road,NaN


## Application Property Town

In [147]:
# how many missing submission dates (including null and white space)
data['application_property_town'].apply(lambda x: pd.isna(x) or str(x).strip() == '').sum()

np.int64(367)

## Source

In [148]:
# Count rows where 'CHAPA - Facebook' appears in the 'source' column (case-insensitive, optional)
chapa_count = data['source'].str.contains('CHAPA Facebook', case=False, na=False).sum()

print(f"Number of entries containing 'CHAPA - Zillow': {chapa_count}")

Number of entries containing 'CHAPA - Zillow': 5


In [149]:
# Count rows where 'CHAPA - Zillow' appears in the 'source' column (case-insensitive, optional)
chapa_count = data['source'].str.contains('CHAPA - Zillow', case=False, na=False).sum()

print(f"Number of entries containing 'CHAPA - Zillow': {chapa_count}")

Number of entries containing 'CHAPA - Zillow': 4


In [150]:
# Count rows where 'CHAPA' appears in the 'source' column (case-insensitive, optional)
chapa_count = data['source'].str.contains('housing', case=False, na=False).sum()

print(f"Number of entries containing 'housing': {chapa_count}")

Number of entries containing 'housing': 6


In [151]:
# Count rows where 'CHAPA' appears in the 'source' column (case-insensitive, optional)
chapa_count = data['source'].str.contains('CHAPA', case=False, na=False).sum()

print(f"Number of entries containing 'CHAPA': {chapa_count}")

Number of entries containing 'CHAPA': 37


In [152]:
# Strip whitespace from all entries in the 'source' column
data['source'] = data['source'].str.strip()
data['source_standardized'] = data['source'].str.lower().replace({
    
    "chapa's list" : 'CHAPA',
    'chapa website' : 'CHAPA',
    'chapa' : 'CHAPA',
    'chapa ready buyer list' : 'CHAPA',
    'ready buyer list' : 'CHAPA',
    'chapa email list' : 'CHAPA',
    'chapa email' : 'CHAPA',
    
    'friends' : 'Friend/Word of Mouth',
    'family' : 'Friend/Word of Mouth',
    'family member' : 'Friend/Word of Mouth',
    'mother' : 'Friend/Word of Mouth',
    'friend' : 'Friend/Word of Mouth',
    'ruth' : 'Friend/Word of Mouth',
    'former resident at groveland comns' : 'Friend/Word of Mouth',
    
    
    'mass housing website' : 'MyMassHome',
    'masshousing' : 'MyMassHome',
    
    'my mass home' : 'MyMassHome',
    'mymasshome' : 'MyMassHome',
    'mymasshomes' : 'MyMassHome',
    'mass homes' : 'MyMassHome',
    'mass.gov' : 'MyMassHome',
    
    'website' : 'Zillow/Trulia/Other Website',
    'online' : 'Zillow/Trulia/Other Website',
    'internet' : 'Zillow/Trulia/Other Website',
    'mls' : 'Zillow/Trulia/Other Website',
    'online research' : 'Zillow/Trulia/Other Website',
    'chapa - zillow' : 'Zillow/Trulia/Other Website',
    'massaccess' : 'Zillow/Trulia/Other Website',
    'urban edge' : 'Zillow/Trulia/Other Website',
    'realtor.com' : 'Zillow/Trulia/Other Website',
    'homes.com' : 'Zillow/Trulia/Other Website',
    'motovo' : 'Zillow/Trulia/Other Website',
    'redfin' : 'Zillow/Trulia/Other Website',
    'zillow' : 'Zillow/Trulia/Other Website',
    '"online research"' : 'Zillow/Trulia/Other Website',
    'bha' : 'Zillow/Trulia/Other Website',

    'boston affordable housing' : 'City of Boston Metrolist',
    'bos.gov' : 'City of Boston Metrolist',
    'boston.gov' : 'City of Boston Metrolist',
    'metrolist' : 'City of Boston Metrolist',
    'metro list' : 'City of Boston Metrolist',
    
    
    'agent' : 'Real Estate Agent',
    'realtor' : 'Real Estate Agent',
    're agent' : 'Real Estate Agent',
    'realtor mls' : 'Real Estate Agent',
    'maloney properites' : 'Real Estate Agent',
    
    'chapa facebook' : 'Social Media',
    'social media' : 'Social Media',
    'facebook' : 'Social Media',
    
    
    'medway mailing list' : 'Local Community',
    'medway housing authority - chapa' : 'Local Community', #could also go under CHAPA
    'andover library' : 'Local Community',
    'canton housing authority ' : 'Local Community',
    'bryna community service network' : 'Local Community',
    'canton paper' : 'Local Community',
    'canton housing authority' : 'Local Community',
    'email list' : 'Local Community'
})

In [153]:
data['source_standardized'].unique()

array([nan, 'Zillow/Trulia/Other Website', 'CHAPA', 'Local Community',
       'Social Media', 'City of Boston Metrolist', 'Friend/Word of Mouth',
       'Real Estate Agent', 'MyMassHome', 'seller is his son'],
      dtype=object)

In [154]:
data['source_standardized'].value_counts()

source_standardized
Zillow/Trulia/Other Website    109
MyMassHome                      58
CHAPA                           29
Real Estate Agent               29
City of Boston Metrolist        16
Friend/Word of Mouth            15
Local Community                 11
Social Media                     9
seller is his son                1
Name: count, dtype: int64

In [155]:
# how many missing sources (including null and white space)
data['source'].apply(lambda x: pd.isna(x) or str(x).strip() == '').sum()

np.int64(103)

In [156]:
# fill in missing vals with "unknown"
data['source_standardized'] = data['source_standardized'].fillna('Unknown')

## Age

In [157]:
#convert to numeric
data['age_numeric'] = pd.to_numeric(data['age'], errors='coerce')

In [158]:
# create new column to flag cols missing age (1 if missing 0 if not missing)
data['age_missing'] = data['age'].isna().astype(int)
data['age_missing'].value_counts()

age_missing
0    287
1     93
Name: count, dtype: int64

In [159]:
age_median = data['age_numeric'].median()
data['age_filled'] = data['age'].fillna(age_median)
data['age_filled'].value_counts()

age_filled
40.0    93
27      18
62      12
41      12
36      12
28      11
30      11
56      10
38       8
35       8
34       8
42       8
57       8
33       8
24       8
29       8
43       7
23       7
44       7
40       7
26       7
31       6
53       6
52       6
69       5
32       5
25       5
64       5
65       4
63       4
59       4
48       4
73       3
76       3
22       3
50       3
67       3
77       3
66       3
39       3
61       3
37       3
49       2
60       2
45       2
46       2
82       1
71       1
55       1
79       1
21       1
68       1
54       1
u/a      1
58       1
74       1
Name: count, dtype: int64

In [160]:
data

,id_number,submission_date,application_property_street_address,application_property_town,source,source_2,age,race/ethnicity,disability,current_residence_town,...,hh_type,hh_income,hh_assets,fthb_class?,unit_number,street_address,source_standardized,age_numeric,age_missing,age_filled
0,150733,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,41,white,NaN,Somerville,...,NaN,"37,934.00",NaN,NaN,NaN,8 Ciderpress Way,Unknown,41.0,0,41
1,439181,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,32,hispanic,NaN,Westford,...,NaN,"61,000.00",NaN,NaN,NaN,8 Ciderpress Way,Unknown,32.0,0,32
2,697399,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,79,black/afr am,NaN,Malden,...,NaN,"31,275.00","346,000.00",NaN,NaN,8 Ciderpress Way,Unknown,79.0,0,79
3,191565,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,NaN,white,NaN,Haverhill,...,NaN,"20,299.00",NaN,NaN,NaN,8 Ciderpress Way,Unknown,NaN,1,40.0
4,226436,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,26,white,NaN,Nashua,...,NaN,"41,000.00",NaN,NaN,NaN,8 Ciderpress Way,Unknown,26.0,0,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,554174,9/26/2023,15 Corn Mill Way,NaN,metrolist,NaN,NaN,Hispanic,NaN,Malden,...,NaN,"85,500.00","48,500.00",NaN,NaN,15 Corn Mill Way,City of Boston Metrolist,NaN,1,40.0
376,377124,9/26/2023,15 Corn Mill Way,NaN,mymasshome,NaN,65,Hispanic,NaN,Scituate,...,NaN,"101,000.00","23,000.00",NaN,NaN,15 Corn Mill Way,MyMassHome,65.0,0,65
377,21605,9/26/2023,15 Corn Mill Way,NaN,mymasshome,NaN,37,white,NaN,holbrook,...,NaN,"62,000.00","74,000.00",NaN,NaN,15 Corn Mill Way,MyMassHome,37.0,0,37
378,56860,9/26/2023,15 Corn Mill Way,NaN,NaN,NaN,48,white,NaN,Quincy,...,NaN,NaN,NaN,NaN,NaN,15 Corn Mill Way,Unknown,48.0,0,48


## Race/Ethnicity

In [161]:
data['race/ethnicity'].value_counts()

race/ethnicity
White                   134
white                    44
Asian                    41
Hispanic                 30
Black                    28
white                    25
hispanic                 13
White                     5
black                     4
african amer              3
asian                     2
hispanic/latino           2
Black/Af Am               2
wihite                    1
hispanic                  1
White/Hispanic            1
asian                     1
north african             1
Cape Ver\dean             1
afr amer                  1
Egyptian                  1
Black                     1
cape vrd blck             1
white/hispanic            1
black/afr amer            1
african/amer              1
native amer/alaskan       1
Other                     1
Hispanic/White            1
Black/White               1
Unknown                   1
South Asian Pakistai      1
Native American           1
white-north african       1
White, Black              1
black

In [162]:
data['race/ethnicity'].unique()

array(['white ', 'hispanic', 'black/afr am', 'White', 'Black', 'Asian',
       'Hispanic', 'White, Black', 'White ', 'Black/Af Am',
       'white-north african', 'Native American', 'white',
       'South Asian Pakistai', 'Unknown', 'Black/White', nan,
       'Hispanic/White', 'Other', 'white/hispanic', 'native amer/alaskan',
       'african/amer', 'hispanic/latino', 'black/afr amer', 'afr amer',
       'cape vrd blck', 'hispanic ', 'black', 'african amer', 'wihite',
       'asian ', 'north african', 'Cape Ver\\dean', 'White/Hispanic',
       'Egyptian', 'Black ', 'asian', 'black/white '], dtype=object)

In [163]:
data['race/ethnicity_CHAPA_groups'] = data['race/ethnicity'].str.lower().str.strip().replace({
    'black/afr am' : 'Black or African American',
    'black/af am' : 'Black or African American',
    'black' : 'Black or African American',
    'white' : 'White/Non-Minority',
    'white-north african' : 'White/Non-Minority',
    'asian' : 'Asian/Pacific Islander',
    'south asian pakistai' : 'Asian/Pacific Islander',
    'native american' : 'Native American/Alaskan Native',
    'native amer/alaskan' : 'Native American/Alaskan Native',
    'african/amer' : 'Black or African American',
    'black/afr amer' : 'Black or African American',
    'afr amer' : 'Black or African American',
    'cape vrd blck' : 'Black or African American',
    'african amer' : 'Black or African American',
    'wihite' : 'White/Non-Minority',
    'north african' : 'North African or Middle Eastern',
    'egyptian' : 'North African or Middle Eastern',
    'white/hispanic' : 'Hispanic/Latino; White/Non-Minority',
    'hispanic/white' : 'Hispanic/Latino; White/Non-Minority',
    'white, black' : 'Black or African American; White/Non-Minority',
    'cape ver\dean' : 'Unknown',
    'unknown' : 'Unknown',
    'hispanic' : 'Hispanic/Latino',
    'hispanic/latino' : 'Hispanic/Latino',
    'black/white' : 'Black or African American; White/Non-Minority'
})

In [164]:
data['race/ethnicity_CHAPA_groups'].value_counts()

race/ethnicity_CHAPA_groups
White/Non-Minority                               210
Hispanic/Latino                                   46
Asian/Pacific Islander                            45
Black or African American                         43
Black or African American; White/Non-Minority      3
Hispanic/Latino; White/Non-Minority                3
Native American/Alaskan Native                     2
Unknown                                            2
North African or Middle Eastern                    2
other                                              1
Name: count, dtype: int64

In [165]:
data['race/ethnicity_CHAPA_groups']

0                                 White/Non-Minority
1                                    Hispanic/Latino
2                          Black or African American
3                                 White/Non-Minority
4                                 White/Non-Minority
                           ...                      
375                                  Hispanic/Latino
376                                  Hispanic/Latino
377                               White/Non-Minority
378                               White/Non-Minority
379    Black or African American; White/Non-Minority
Name: race/ethnicity_CHAPA_groups, Length: 380, dtype: object

In [166]:
data['race/ethnicity_CHAPA_groups'] = data['race/ethnicity_CHAPA_groups'].fillna("Unknown")
data['race/ethnicity_CHAPA_groups'].value_counts()

race/ethnicity_CHAPA_groups
White/Non-Minority                               210
Hispanic/Latino                                   46
Asian/Pacific Islander                            45
Black or African American                         43
Unknown                                           25
Black or African American; White/Non-Minority      3
Hispanic/Latino; White/Non-Minority                3
Native American/Alaskan Native                     2
North African or Middle Eastern                    2
other                                              1
Name: count, dtype: int64

In [167]:
data['hispanic/latino_census'] = data['race/ethnicity_CHAPA_groups'].str.contains('Hispanic/Latino', case=False, na=False).astype(int)
data['hispanic/latino_census'].value_counts()

hispanic/latino_census
0    331
1     49
Name: count, dtype: int64

In [168]:
data['race_census'] = data['race/ethnicity_CHAPA_groups'].replace({
    'White/Non-Minority' : 'White alone',
    'Hispanic/Latino' : 'Unknown',
    'Asian/Pacific Islander' : 'Asian alone',
    'Black or African American' : 'Black or African American alone',
    'Black or African American; White/Non-Minority' : 'White; Black or African American',
    'Hispanic/Latino; White/Non-Minority' : 'White alone',
    'Native American/Alaskan Native' : 'American Indian and Alaska Native alone',
    'North African or Middle Eastern' : 'White alone',
    'other' : 'Some Other Race alone'
})
data['race_census'].value_counts()

race_census
White alone                                215
Unknown                                     71
Asian alone                                 45
Black or African American alone             43
White; Black or African American             3
American Indian and Alaska Native alone      2
Some Other Race alone                        1
Name: count, dtype: int64

## Disability

In [169]:
data['disability'].unique()

array([nan, 'Disabled', 'no', 'Yes'], dtype=object)

In [170]:
data['disability'].value_counts()

disability
no          16
Disabled     2
Yes          1
Name: count, dtype: int64

In [171]:
data['disability_standardized'] = data['disability'].str.lower().str.strip().replace({
    'disabled' : 1,
    'yes' : 1,
    'no' : 0
})


/var/folders/qm/352493r135l4rqs1sw0gsnyr0000gn/T/ipykernel_24499/3370909249.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['disability_standardized'] = data['disability'].str.lower().str.strip().replace({


In [172]:
print(data['disability_standardized'].isna().sum())

361


In [173]:
data['disability_standardized'].value_counts()

disability_standardized
0.0    16
1.0     3
Name: count, dtype: int64

In [174]:

# fill all null values with "N"
data['disability_standardized'] = data['disability_standardized'].replace(['nan', '', 'None', 'N/A'], pd.NA)
data['disability_standardized'] = data['disability_standardized'].fillna(0)
 

In [175]:
print(data['disability_standardized'].isna().sum())

0


In [176]:
data['disability_standardized'].value_counts()

disability_standardized
0.0    377
1.0      3
Name: count, dtype: int64

## Current Residence Town

In [177]:
# fill missing with "unknown"
data['current_residence_town_standardized'] = data['current_residence_town'].str.lower().str.strip().replace(['nh', '', 'nan', '', 'None', 'N/A'], 'unknown')
data['current_residence_town_standardized'] = data['current_residence_town_standardized'].fillna("unknown")

In [178]:
data['current_residence_town_standardized'].value_counts()

current_residence_town_standardized
andover            11
quincy             11
medway             11
boston             10
taunton            10
                   ..
kensington          1
townsend            1
north billerica     1
plaistow            1
holbrook            1
Name: count, Length: 139, dtype: int64

In [179]:
data['current_residence_town_standardized'].unique()

array(['somerville', 'westford', 'malden', 'haverhill', 'nashua',
       'east boston', 'waltham', 'woburn', 'burlington', 'chelmsford',
       'mansfield', 'taunton', 'beverly', 'billerica', 'lawrence',
       'reading', 'tyngsboro', 'bedford', 'ipswich', 'middleborough',
       'stoneham', 'saugus', 'north andover', 'dracut', 'andover',
       'cape coral', 'topsfield', 'chester', 'north reading', 'revere',
       'bellingham', 'north attleboro', 'plainville', 'medway',
       'uxbridge', 'franklin', 'ashland', 'newburyport', 'watertown',
       'dorchester', 'peabody', 'salem', 'burlignton', 'derry',
       'stoughton', 'canton', 'hyde park', 'quincy', 'litchfield',
       'westminster', 'everett', 'lowell', 'hopewell', 'marblehead',
       'boston', 'melrose', 'swampscott', 'wilmington', 'stow',
       'arlignton', 'tewksbury', 'north billerica', 'wakefield',
       'roslindale', 'weymouth', 'unknown', 'medford', 'foxboro',
       'brockton', 'west roxbury', 'norwood', 'dedham', 'w

In [180]:
data['current_residence_town_standardized'] = data['current_residence_town_standardized'].str.strip().str.title()

## Dependents 

In [181]:
# create new column to flag cols missing dependents (1 if missing 0 if not missing)
data['dependents_missing'] = data['dependents'].isna().astype(int)
data['dependents_missing'].value_counts()

dependents_missing
0    262
1    118
Name: count, dtype: int64

In [182]:
data

,id_number,submission_date,application_property_street_address,application_property_town,source,source_2,age,race/ethnicity,disability,current_residence_town,...,source_standardized,age_numeric,age_missing,age_filled,race/ethnicity_CHAPA_groups,hispanic/latino_census,race_census,disability_standardized,current_residence_town_standardized,dependents_missing
0,150733,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,41,white,NaN,Somerville,...,Unknown,41.0,0,41,White/Non-Minority,0,White alone,0.0,Somerville,1
1,439181,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,32,hispanic,NaN,Westford,...,Unknown,32.0,0,32,Hispanic/Latino,1,Unknown,0.0,Westford,1
2,697399,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,79,black/afr am,NaN,Malden,...,Unknown,79.0,0,79,Black or African American,0,Black or African American alone,0.0,Malden,1
3,191565,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,NaN,white,NaN,Haverhill,...,Unknown,NaN,1,40.0,White/Non-Minority,0,White alone,0.0,Haverhill,1
4,226436,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,26,white,NaN,Nashua,...,Unknown,26.0,0,26,White/Non-Minority,0,White alone,0.0,Nashua,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,554174,9/26/2023,15 Corn Mill Way,NaN,metrolist,NaN,NaN,Hispanic,NaN,Malden,...,City of Boston Metrolist,NaN,1,40.0,Hispanic/Latino,1,Unknown,0.0,Malden,0
376,377124,9/26/2023,15 Corn Mill Way,NaN,mymasshome,NaN,65,Hispanic,NaN,Scituate,...,MyMassHome,65.0,0,65,Hispanic/Latino,1,Unknown,0.0,Scituate,0
377,21605,9/26/2023,15 Corn Mill Way,NaN,mymasshome,NaN,37,white,NaN,holbrook,...,MyMassHome,37.0,0,37,White/Non-Minority,0,White alone,0.0,Holbrook,0
378,56860,9/26/2023,15 Corn Mill Way,NaN,NaN,NaN,48,white,NaN,Quincy,...,Unknown,48.0,0,48,White/Non-Minority,0,White alone,0.0,Quincy,0


In [183]:
data['dependents'].unique()

array([nan,  1.,  0.,  2.,  3.])

In [184]:
# fill missing dependents with median
dependent_median = data['dependents'].median()
data['dependents_filled'] = data['dependents'].fillna(dependent_median)
data['dependents_filled'].value_counts()

dependents_filled
0.0    277
1.0     60
2.0     33
3.0     10
Name: count, dtype: int64

## Household Income

In [185]:
data['hh_income'].unique()

array(['37,934.00', '61,000.00', '31,275.00', '20,299.00', '41,000.00',
       '68,500.00', '43,000.00', '50,000.00', '68,400.00', '72,000.00',
       '40,000.00', '58,000.00', '45,000.00', '37,400.00', '53,000.00',
       '46,500.00', '51,000.00', '62,500.00', '49,000.00', '42,500.00',
       '38,000.00', '53,500.00', '38,400.00', '52,000.00', '49,313.00',
       '54,000.00', '47,000.00', '76,000.00', '70,000.00', '33,000.00',
       '48,000.00', '44,000.00', '47,200.00', '22,000.00', '11,000.00',
       '40,300.00', '39,000.00', '37,814.00', '49,763.00', '74,000.00',
       '30,780.00', '83,000.00', '57,000.00', '56,000.00', '35,000.00',
       '55,000.00', '65,000.00', '67,000.00', '78,000.00', '90,000.00',
       '61,500.00', '62,000.00', '46,000.00', '110,000.00', '92,082.00',
       '73,937.00', '75,485.00', '42,000.00', '91,000.00', '60,000.00',
       '$20,000.00', '66,000.00', '57,076.00', '89,000.00', '67,636.00',
       '24,000.00', '19,044.00', '81,497.00', '78,144.00', '95

In [186]:
#convert to numeric
data['hh_income_numeric'] = data['hh_income'].str.replace(',', '')
data['hh_income_numeric'] = pd.to_numeric(data['hh_income_numeric'], errors='coerce')

In [187]:
# create new column to flag cols missing HH income (1 if missing 0 if not missing)
data['hh_income_missing'] = data['hh_income_numeric'].isna().astype(int)
data['hh_income_missing'].value_counts()

hh_income_missing
0    377
1      3
Name: count, dtype: int64

In [188]:
hh_income_median = data['hh_income_numeric'].median()
data['hh_income_filled'] = data['hh_income_numeric'].fillna(hh_income_median)
data['hh_income_filled'].value_counts()
data['hh_income_filled']

0       37934.0
1       61000.0
2       31275.0
3       20299.0
4       41000.0
         ...   
375     85500.0
376    101000.0
377     62000.0
378     60000.0
379     68000.0
Name: hh_income_filled, Length: 380, dtype: float64

In [189]:
# creating income brackets

# Define the max income to create the bins
max_income = data['hh_income_filled'].max()

# Create bins: from 0 to the next 20k above max_income, in steps of 20k
bins = list(range(0, int(max_income + 20000), 20000))

# Create labels for each bin
labels = [f"${bins[i]}–${bins[i+1]-1}" for i in range(len(bins)-1)]

# Create the bracket column
data['income_bracket'] = pd.cut(data['hh_income_filled'], bins=bins, labels=labels, right=False)


In [190]:
data[['hh_income_filled', 'income_bracket']].head()

,hh_income_filled,income_bracket
0,37934.0,$20000–$39999
1,61000.0,$60000–$79999
2,31275.0,$20000–$39999
3,20299.0,$20000–$39999
4,41000.0,$40000–$59999


## Household Assets

In [191]:
#convert to numeric
data['hh_assets_numeric'] = data['hh_assets'].str.replace(',', '')
data['hh_assets_numeric'] = pd.to_numeric(data['hh_assets_numeric'], errors='coerce')

In [192]:
# create new column to flag cols missing HH income (1 if missing 0 if not missing)
data['hh_assets_missing'] = data['hh_assets_numeric'].isna().astype(int)
data['hh_assets_missing'].value_counts()

hh_assets_missing
0    357
1     23
Name: count, dtype: int64

In [193]:
hh_assets_median = data['hh_assets_numeric'].median()
data['hh_assets_filled'] = data['hh_assets_numeric'].fillna(hh_assets_median)
data['hh_assets_filled'].value_counts()
data['hh_assets_filled']

0       36000.0
1       36000.0
2      346000.0
3       36000.0
4       36000.0
         ...   
375     48500.0
376     23000.0
377     74000.0
378     36000.0
379      9865.0
Name: hh_assets_filled, Length: 380, dtype: float64

In [194]:
data

,id_number,submission_date,application_property_street_address,application_property_town,source,source_2,age,race/ethnicity,disability,current_residence_town,...,current_residence_town_standardized,dependents_missing,dependents_filled,hh_income_numeric,hh_income_missing,hh_income_filled,income_bracket,hh_assets_numeric,hh_assets_missing,hh_assets_filled
0,150733,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,41,white,NaN,Somerville,...,Somerville,1,0.0,37934.0,0,37934.0,$20000–$39999,NaN,1,36000.0
1,439181,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,32,hispanic,NaN,Westford,...,Westford,1,0.0,61000.0,0,61000.0,$60000–$79999,NaN,1,36000.0
2,697399,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,79,black/afr am,NaN,Malden,...,Malden,1,0.0,31275.0,0,31275.0,$20000–$39999,346000.0,0,346000.0
3,191565,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,NaN,white,NaN,Haverhill,...,Haverhill,1,0.0,20299.0,0,20299.0,$20000–$39999,NaN,1,36000.0
4,226436,2/8/2022,8 Ciderpress Way,NaN,NaN,NaN,26,white,NaN,Nashua,...,Nashua,1,0.0,41000.0,0,41000.0,$40000–$59999,NaN,1,36000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,554174,9/26/2023,15 Corn Mill Way,NaN,metrolist,NaN,NaN,Hispanic,NaN,Malden,...,Malden,0,2.0,85500.0,0,85500.0,$80000–$99999,48500.0,0,48500.0
376,377124,9/26/2023,15 Corn Mill Way,NaN,mymasshome,NaN,65,Hispanic,NaN,Scituate,...,Scituate,0,1.0,101000.0,0,101000.0,$100000–$119999,23000.0,0,23000.0
377,21605,9/26/2023,15 Corn Mill Way,NaN,mymasshome,NaN,37,white,NaN,holbrook,...,Holbrook,0,0.0,62000.0,0,62000.0,$60000–$79999,74000.0,0,74000.0
378,56860,9/26/2023,15 Corn Mill Way,NaN,NaN,NaN,48,white,NaN,Quincy,...,Quincy,0,0.0,NaN,1,60000.0,$60000–$79999,NaN,1,36000.0


In [195]:
data['hh_assets_filled'].describe()

count       380.000000
mean      49038.331579
std       54634.158747
min           0.000000
25%       19072.750000
50%       36000.000000
75%       60000.000000
max      346000.000000
Name: hh_assets_filled, dtype: float64

In [196]:
asset_bins = [0, 5000, 20000, 40000, 60000, 100000, 200000, float('inf')]
asset_labels = [
    '<$5k', '$5k–20k', '$20k–40k', '$40k–60k',
    '$60k–100k', '$100k–200k', '$200k+'
]

data['asset_bracket'] = pd.cut(
    data['hh_assets_filled'],
    bins=asset_bins,
    labels=asset_labels,
    right=False  # optional: include left edge, exclude right
)

data[['hh_assets_filled', 'asset_bracket']].head()

,hh_assets_filled,asset_bracket
0,36000.0,$20k–40k
1,36000.0,$20k–40k
2,346000.0,$200k+
3,36000.0,$20k–40k
4,36000.0,$20k–40k


# Save Cleaned Data

In [197]:
data.to_csv('/Users/kaylamullen/Desktop/PITNE/Data/cleaned_data.csv', index=False)